# TCGA-BRCA TumorArea-Coarse Mask Visualisation

Visualise tumor inclusion/exclusion polygon annotations from:
https://github.com/DeepMicroscopy/TCGA-BRCA-TumorArea-Coarse

**Setup:**
1. Clone the annotations repo and note the path to the `.p` file
2. Download TCGA-BRCA SVS slides from GDC Data Portal
3. Update `CONFIG` paths below

In [ ]:
import pickle
import openslide
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from PIL import Image
import os
from pathlib import Path

## Config

In [ ]:
CONFIG = {
    # Path to the .p annotations file from the TCGA-BRCA-TumorArea-Coarse repo
    "annotations_p": "tcga_brca_annotations.p",
    
    # Directory containing downloaded SVS slides
    "slides_dir": ".datasets/TCGA-BRCA",
    
    # Downsample factor used when creating annotations (16 = level-of-detail)
    "annotation_downsample": 16,
    
    # Thumbnail size for display
    "thumbnail_size": (1024, 1024),
    
    # Output directory for saved figures
    "output_dir": ".scratch/tcga_brca_masks",
}

## Load Annotations

In [ ]:
def load_annotations(p_file_path: str):
    """Load .p pickle file with incl_vec/excl_vec polygon dicts."""
    with open(p_file_path, 'rb') as f:
        ann = pickle.load(f)
    return ann['incl_vec'], ann['excl_vec']


incl_polys, excl_polys = load_annotations(CONFIG["annotations_p"])

print(f"Slides with inclusion annotations: {len(incl_polys)}")
print(f"Slides with exclusion annotations: {len(excl_polys)}")
print("\nFirst 5 slide keys:")
for k in list(incl_polys.keys())[:5]:
    n_incl = len(incl_polys.get(k, []))
    n_excl = len(excl_polys.get(k, []))
    print(f"  {k}  →  incl={n_incl}  excl={n_excl}")

## Discover Available SVS Slides

In [ ]:
slides_dir = Path(CONFIG["slides_dir"])
svs_files = sorted(slides_dir.rglob("*.svs"))
print(f"Found {len(svs_files)} SVS files")

# Match SVS files to annotation keys (TIFF keys have .tif extension)
matched = []
for svs in svs_files:
    tiff_key = svs.stem + ".tif"
    if tiff_key in incl_polys:
        matched.append((svs, tiff_key))

print(f"Matched {len(matched)} slides to annotations")
for svs, key in matched[:10]:
    print(f"  {svs.name}")

## Helper: Scale Polygons to Thumbnail Coords

Annotation coordinates are at `annotation_downsample` × lower resolution than the level-0 slide dimensions. We need to scale them to match the thumbnail pixel coordinates.

In [ ]:
def scale_polygons(polys, slide, thumbnail_size, annotation_downsample):
    """
    Scale polygon coords from annotation space to thumbnail pixel space.
    
    Annotation coords are at (level-0 / annotation_downsample) resolution.
    Thumbnail is a fixed-size render of the full slide.
    """
    w0, h0 = slide.dimensions  # level-0 dimensions
    tw, th = thumbnail_size
    
    # Actual thumbnail size respects aspect ratio
    aspect = w0 / h0
    if aspect > 1:
        tw_actual, th_actual = tw, int(tw / aspect)
    else:
        tw_actual, th_actual = int(th * aspect), th
    
    sx = tw_actual / (w0 / annotation_downsample)
    sy = th_actual / (h0 / annotation_downsample)
    
    scaled = []
    for poly in polys:
        pts = np.array(poly)
        pts[:, 0] *= sx
        pts[:, 1] *= sy
        scaled.append(pts)
    return scaled

## Visualise a Single Slide

In [ ]:
def visualise_slide(slide_path, tiff_key, incl_polys, excl_polys,
                    annotation_downsample=16, thumbnail_size=(1024, 1024),
                    save_path=None, ax=None):
    """
    Overlay tumor inclusion (green) and exclusion (red) polygons on WSI thumbnail.
    """
    slide = openslide.OpenSlide(str(slide_path))
    thumb = slide.get_thumbnail(thumbnail_size).convert('RGB')

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(10, 10))

    ax.imshow(thumb)
    ax.set_title(tiff_key, fontsize=9)
    ax.axis('off')

    incl = incl_polys.get(tiff_key, [])
    excl = excl_polys.get(tiff_key, [])

    incl_scaled = scale_polygons(incl, slide, thumbnail_size, annotation_downsample)
    excl_scaled = scale_polygons(excl, slide, thumbnail_size, annotation_downsample)

    for pts in incl_scaled:
        patch = Polygon(pts, closed=True, alpha=0.35,
                        facecolor='limegreen', edgecolor='darkgreen', lw=1.2)
        ax.add_patch(patch)

    for pts in excl_scaled:
        patch = Polygon(pts, closed=True, alpha=0.35,
                        facecolor='red', edgecolor='darkred', lw=1.2)
        ax.add_patch(patch)

    legend_handles = [
        mpatches.Patch(facecolor='limegreen', edgecolor='darkgreen', alpha=0.6,
                       label=f'Tumor (incl) n={len(incl_scaled)}'),
        mpatches.Patch(facecolor='red', edgecolor='darkred', alpha=0.6,
                       label=f'Excluded n={len(excl_scaled)}'),
    ]
    ax.legend(handles=legend_handles, loc='lower right', fontsize=8)

    if standalone:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved → {save_path}")
        plt.show()

    slide.close()

In [ ]:
# Visualise the first matched slide
if matched:
    svs_path, tiff_key = matched[0]
    visualise_slide(
        svs_path, tiff_key,
        incl_polys, excl_polys,
        annotation_downsample=CONFIG["annotation_downsample"],
        thumbnail_size=CONFIG["thumbnail_size"],
    )
else:
    print("No matched slides found. Check slides_dir and annotations_p paths.")

## Visualise Multiple Slides in a Grid

In [ ]:
def visualise_grid(matched_slides, incl_polys, excl_polys,
                   annotation_downsample=16, thumbnail_size=(512, 512),
                   ncols=3, save_path=None):
    n = len(matched_slides)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 5))
    axes = np.array(axes).flatten()

    for i, (svs_path, tiff_key) in enumerate(matched_slides):
        visualise_slide(
            svs_path, tiff_key, incl_polys, excl_polys,
            annotation_downsample=annotation_downsample,
            thumbnail_size=thumbnail_size,
            ax=axes[i],
        )

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("TCGA-BRCA TumorArea-Coarse Annotations", fontsize=14, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"Saved → {save_path}")
    plt.show()


# Show first 6 matched slides
if matched:
    visualise_grid(
        matched[:6], incl_polys, excl_polys,
        annotation_downsample=CONFIG["annotation_downsample"],
        thumbnail_size=(512, 512),
        ncols=3,
    )

## Generate and Save Binary Masks

Rasterise inclusion polygons into a binary mask at the annotation resolution, then optionally upsample to a target level.

In [ ]:
from PIL import ImageDraw


def make_binary_mask(slide_path, tiff_key, incl_polys, excl_polys,
                     annotation_downsample=16):
    """
    Rasterise inclusion polygons into a binary mask at annotation resolution.
    Exclusion polygons are set to 0 inside inclusion regions.

    Returns:
        mask (np.ndarray, uint8): H x W binary mask (255 = tumor, 0 = background)
        mask_size (tuple): (W, H) at annotation resolution
    """
    slide = openslide.OpenSlide(str(slide_path))
    w0, h0 = slide.dimensions
    slide.close()

    W = int(np.ceil(w0 / annotation_downsample))
    H = int(np.ceil(h0 / annotation_downsample))

    mask_img = Image.new('L', (W, H), 0)
    draw = ImageDraw.Draw(mask_img)

    for poly in incl_polys.get(tiff_key, []):
        pts = [(float(x), float(y)) for x, y in poly]
        draw.polygon(pts, fill=255)

    for poly in excl_polys.get(tiff_key, []):
        pts = [(float(x), float(y)) for x, y in poly]
        draw.polygon(pts, fill=0)

    return np.array(mask_img), (W, H)


# Demo mask for first slide
if matched:
    svs_path, tiff_key = matched[0]
    mask, mask_size = make_binary_mask(
        svs_path, tiff_key, incl_polys, excl_polys,
        annotation_downsample=CONFIG["annotation_downsample"],
    )
    print(f"Mask shape: {mask.shape}  Tumor pixels: {(mask > 0).sum():,}  ({100*(mask>0).mean():.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    slide = openslide.OpenSlide(str(svs_path))
    thumb = slide.get_thumbnail(CONFIG["thumbnail_size"]).convert('RGB')
    slide.close()

    axes[0].imshow(thumb)
    axes[0].set_title('Thumbnail')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='Greens', vmin=0, vmax=255)
    axes[1].set_title(f'Binary Tumor Mask ({mask_size[0]}×{mask_size[1]})')
    axes[1].axis('off')

    plt.suptitle(tiff_key, fontsize=10)
    plt.tight_layout()
    plt.show()

## Batch Save Masks

In [ ]:
out_dir = Path(CONFIG["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

for svs_path, tiff_key in matched:
    stem = Path(tiff_key).stem
    mask_path = out_dir / f"{stem}_mask.png"
    overlay_path = out_dir / f"{stem}_overlay.png"

    # Save binary mask
    mask, _ = make_binary_mask(
        svs_path, tiff_key, incl_polys, excl_polys,
        annotation_downsample=CONFIG["annotation_downsample"],
    )
    Image.fromarray(mask).save(mask_path)

    # Save overlay figure
    visualise_slide(
        svs_path, tiff_key, incl_polys, excl_polys,
        annotation_downsample=CONFIG["annotation_downsample"],
        thumbnail_size=CONFIG["thumbnail_size"],
        save_path=str(overlay_path),
    )
    plt.close('all')
    print(f"  {stem} done")

print(f"\nAll outputs saved to {out_dir}")